# Local Setup Instructions

**This notebook is configured for local execution (not Google Colab).**

## Prerequisites

1. **Install packages**:
   ```bash
   pip install earthengine-api geemap geopandas shapely pandas numpy tqdm
   ```

2. **Authenticate Earth Engine** (one-time):
   ```bash
   earthengine authenticate
   ```

## Important Notes

- **Exports go to Google Drive**: Earth Engine batch exports are sent to your Google Drive (root folder by default).
- After exports complete, **download the CSVs** to: `{OUT_DIR}/DRIVE_CLIMATE_BY_HSA_DOWNLOAD/FINAL_HSA_CLIMATE/`
- Adjust `GEOJSON_PATH` in STEP 1.1 to point to your HSA boundaries file.

---


# HSAs → Weekly Climate Features with 20‑Day Lags (Daily‑Only Sources)


**Datasets**
- Precipitation: CHIRPS Daily
- Temperature, Dewpoint, Wind: ERA5‑Land Hourly → daily aggregates
- Evaporation proxy: ERA5‑Land Surface Latent Heat Flux → mm/day (hourly→daily)
- Soil Moisture layers (1–4): ERA5‑Land Hourly → daily aggregates
- Elevation: SRTM (static)

**Notes**
- Keys HSAs by `anchor_name`.
- Uses polygon‑by‑polygon exports to avoid geometry batching errors.
- TEST MODE allows running a small subset first.


## STEP 0.1 — Install and Initialize Earth Engine (with Cloud Project)

In [ ]:
# ── BOUNDARY VERSION SELECTOR ──────────────────────────────────────────────
# Choose which HSA boundary bundle to use for this analysis.
# Must match a bundle produced by HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v7"   # change as needed
# ────────────────────────────────────────────────────────────────────────────


In [ ]:
# @title STEP 0.1 — Initialize Earth Engine
# Install via: pip install earthengine-api geemap geopandas shapely pandas numpy tqdm
# First time: run 'earthengine authenticate' in terminal

import ee
# CHANGE THIS to your own GEE project ID (see SETUP_INSTRUCTIONS.md)
PROJECT = "ee-izaslavsky"  # <-- replace with your project ID

try:
    ee.Initialize(project=PROJECT)
except Exception:
    print('Authentication required. Run: earthengine authenticate')
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print(f"✓ Earth Engine initialized with project: {PROJECT}")

## STEP 0.2 — Configuration (dates, scale, toggles, test mode)

In [ ]:

# @title STEP 0.2 — Config
from datetime import date, timedelta

WEEK_START = "2022-06-27"
# WEEK_START = "2019-01-07"
WEEK_END   = "2024-01-29"
MAX_LAG_DAYS = 20

NETWORK = "INF"  # @param ["INF", "NCD"]
MODE="fewest"

# Set data and output directory
DATA_DIR = "data"
OUT_DIR = os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", "out"))

TEST_MODE = True
TEST_HSA_COUNT  = 5
TEST_WEEK_COUNT = 3

USE_CHIRPS        = True
USE_ERA5_HOURLY   = True
USE_ERA5_EVP      = True
INCLUDE_ELEVATION = True

COLL = {
    "CHIRPS_DAILY":     "UCSB-CHG/CHIRPS/DAILY",
    "ERA5_LAND_HOURLY": "ECMWF/ERA5_LAND/HOURLY",
    "SRTM":             "USGS/SRTMGL1_003",
}

BANDS = {
    "CHIRPS": {"precip": "precipitation"},
    "ERA5": {
        "t_mean": "temperature_2m",
        "td":     "dewpoint_temperature_2m",
        "u10":    "u_component_of_wind_10m",
        "v10":    "v_component_of_wind_10m",
        "lhf":    "surface_latent_heat_flux",
        "swvl1":  "volumetric_soil_water_layer_1",
        "swvl2":  "volumetric_soil_water_layer_2",
        "swvl3":  "volumetric_soil_water_layer_3",
        "swvl4":  "volumetric_soil_water_layer_4",
    },
}

SCALE = {"CHIRPS": 5550, "ERA5": 9000, "ELEV": 30}

print(f"✓ Config set (NETWORK={NETWORK}, BOUNDARY_VERSION={BOUNDARY_VERSION})")


## STEP 1.1 — Load HSA GeoJSON (must contain `anchor_name`)

In [ ]:
# @title STEP 1.1 — Load HSA GeoJSON (local path)
import geopandas as gpd
import os

# LOCAL FILE PATH - adjust as needed
GEOJSON_PATH = os.path.join(OUT_DIR, f"{NETWORK}_{MODE}_hsas_{BOUNDARY_VERSION}.geojson")

print(f'Loading from: {GEOJSON_PATH}')
gdf = gpd.read_file(GEOJSON_PATH)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)

id_col = "FacilityName"
assert id_col in gdf.columns, f"GeoJSON must include a '{id_col}' column."

gdf = gdf[~gdf.geometry.is_empty].copy()
print(f"✓ Loaded {len(gdf)} HSAs; using id_col = '{id_col}'")

## STEP 1.2 — Strict Shapely→EE geometry converter (rings only)

In [ ]:
# ============================================
# STEP 1.2 — Build EE geometries from GeoDataFrame (server-side, robust)
# ============================================

import ee, geemap

id_col = "FacilityName"
assert id_col in gdf.columns, f"Missing '{id_col}' in GeoJSON."

# Make sure id is string (safer for Filters)
gdf[id_col] = gdf[id_col].astype(str)

# Convert entire GeoDataFrame to an EE FeatureCollection
# geodesic=False to avoid topology surprises; keeps properties including anchor_name
ee_fc_hsa = geemap.gdf_to_ee(gdf[[id_col, "geometry"]], geodesic=False)

# Quick server-side sanity: count & list a few ids
print("EE HSAs count:", ee_fc_hsa.size().getInfo())
print("First 5 anchor_name values:",
      ee_fc_hsa.limit(5).reduceColumns(ee.Reducer.toList(), [id_col]).get("list").getInfo())

# Accessor: fetch an EE Geometry for a given anchor_name directly from EE (no Python reconstruction)
def ee_geom_by_id(hid: str) -> ee.Geometry:
    f = ee_fc_hsa.filter(ee.Filter.eq(id_col, hid)).first()
    # Force early failure if not found or invalid
    g = ee.Algorithms.If(f, ee.Feature(f).geometry(), None)
    g = ee.Geometry(g)
    _ = g.type().getInfo()   # validate server-side now
    return g

# Rebuild HSA_LIST / HSA_BY_ID using the server-side features (no local GeoJSON construction)
HSA_LIST, HSA_BY_ID, HSA_FALLBACK_LOG = [], {}, []
ids = gdf[id_col].tolist()

for hid in ids:
    try:
        geom = ee_geom_by_id(hid)
        HSA_LIST.append({"id": hid, "geom": geom, "fallback": "ee_feature_geometry"})
        HSA_BY_ID[hid] = {"id": hid, "geom": geom, "fallback": "ee_feature_geometry"}
        HSA_FALLBACK_LOG.append((hid, "ee_feature_geometry"))
    except Exception as e:
        print(f"❌ {hid}: {e}")
        HSA_FALLBACK_LOG.append((hid, f"failed:{e}"))

print(f"HSAs converted: {len(HSA_LIST)}")
print("Examples:", HSA_FALLBACK_LOG[:5])

# OPTIONAL: one-polygon micro-test (CHIRPS weekly mean) to prove geometry works before exporting
_test_id = ids[0]
_test_week = "2023-01-02"  # rainy season week in Jordan
_g = HSA_BY_ID[_test_id]["geom"]
start = ee.Date(_test_week)
img = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
       .filterDate(start, start.advance(7, 'day'))
       .select("precipitation")
       .sum())
print(f"CHIRPS sum (mm) for '{_test_id}' week {_test_week}:",
      img.reduceRegion(ee.Reducer.mean(), _g, 5550, maxPixels=1e9).getInfo())


## STEP 2.1 — Compute Monday‑anchored weeks

In [ ]:
# STEP 2.1 — Build anchor_mondays (all Mondays, inclusive)
from datetime import date, timedelta

START_ISO = '2022-06-27'  # Monday
END_ISO   = '2024-01-29'  # Monday

start = date.fromisoformat(START_ISO)
end   = date.fromisoformat(END_ISO)

# sanity: both endpoints must be Monday (Monday=0)
assert start.weekday() == 0 and end.weekday() == 0, "Start/End must be Mondays"

anchor_mondays = []
d = start
while d <= end:
    anchor_mondays.append(d.isoformat())
    d += timedelta(days=7)

print(f"anchor_mondays: {len(anchor_mondays)} weeks ({anchor_mondays[0]} → {anchor_mondays[-1]})")


## STEP 3 — Feature families (weekly + d‑1..d‑20)

In [ ]:
def family_precip_week_and_lags_IMPROVED(fc, week_iso):
    """Enhanced with cumulative windows"""

    if not USE_CHIRPS:
        return fc
    start = ee.Date(week_iso)
    week_str = ee.Date(week_iso).format('YYYY-MM-dd')

    ic = (ee.ImageCollection(COLL["CHIRPS_DAILY"])
          .filterDate(start, start.advance(7,'day'))
          .select("precipitation")
          .map(lambda im: im.unmask(0)))

    # EXISTING: Weekly features
    weekly = ee.Image().addBands([
        ic.mean().rename('P_mean_week'),
        ic.sum().rename('P_total_week'),
        ic.map(lambda im: im.gt(1.0)).mean().rename('wetday_frac_week'),
        ic.map(lambda im: im.gt(20.0)).sum().rename('heavy_days_week'),
        ic.reduce(ee.Reducer.percentile([95])).rename(['P95_week']),
        ic.reduce(ee.Reducer.max()).rename('P_max_day_week'),  # NEW: Max daily
    ])

    # NEW: Cumulative lag windows (reduce multicollinearity)
    lag_windows = []

    # 1-week lags (3 windows)
    for w in [1, 2, 3]:
        d0 = start.advance(-w*7, 'day')
        d1 = d0.advance(7, 'day')
        img = (ee.ImageCollection(COLL["CHIRPS_DAILY"])
               .filterDate(d0, d1)
               .select("precipitation")
               .map(lambda im: im.unmask(0)))

        lag_windows.append(img.sum().rename(f'P_sum_lag_w-{w}'))
        lag_windows.append(img.mean().rename(f'P_mean_lag_w-{w}'))
        lag_windows.append(img.map(lambda im: im.gt(10)).sum().rename(f'P_heavy_days_lag_w-{w}'))

    # OPTIONAL: Keep key daily lags (reduce from 20 to 7 for parsimony)
    daily_lags = []
    for k in [1, 2, 3, 5, 7, 10, 14]:  # Selected days only
        d0 = start.advance(-k,'day')
        di = (ee.ImageCollection(COLL["CHIRPS_DAILY"])
              .filterDate(d0, d0.advance(1,'day'))
              .select("precipitation")
              .map(lambda im: im.unmask(0))
              .mean()
              .rename(f'P_d-{k}'))
        daily_lags.append(di)

    # Combine
    img = weekly.addBands(ee.ImageCollection(lag_windows + daily_lags).toBands())

    def reduce_one(f):
        r = img.reduceRegion(ee.Reducer.mean(), f.geometry(), SCALE["CHIRPS"], maxPixels=1e9)
        return f.set(r).set({'week_start': week_str})

    return fc.map(reduce_one)


In [ ]:
def _daily_td_w_IMPROVED(date_iso):
    """Add heat index and stress metrics"""
    d0 = ee.Date(date_iso); d1 = d0.advance(1,'day')
    day = ee.ImageCollection(COLL["ERA5_LAND_HOURLY"]).filterDate(d0,d1)

    t  = day.select("temperature_2m")
    td = day.select("dewpoint_temperature_2m")
    u  = day.select("u_component_of_wind_10m")
    v  = day.select("v_component_of_wind_10m")

    # Existing
    t_mean = t.mean().subtract(273.15).rename('T_mean_C')
    t_min  = t.min().subtract(273.15).rename('T_min_C')
    t_max  = t.max().subtract(273.15).rename('T_max_C')
    td_c   = td.mean().subtract(273.15).rename('Td_C')
    wspd   = u.mean().hypot(v.mean()).rename('wind_speed_ms')

    # NEW: Diurnal temperature range
    dtr = t.max().subtract(t.min()).rename('DTR_C')

    # NEW: Heat stress hours
    t_celsius = t.subtract(273.15)
    hours_above_30 = t_celsius.map(lambda img: img.gt(30)).sum().rename('hours_above_30C')
    hours_above_35 = t_celsius.map(lambda img: img.gt(35)).sum().rename('hours_above_35C')

    # NEW: Simplified heat index (approximation)
    # HI ≈ T + 0.5555 * (e - 10), where e = vapor pressure from dewpoint
    # Simplified: HI ≈ T_mean + 0.4 * (Td - T_mean)  for quick estimate
    t_mean_k = t.mean()
    td_mean_k = td.mean()
    heat_index_approx = t_mean_k.add(td_mean_k.subtract(t_mean_k).multiply(0.4)).subtract(273.15).rename('heat_index_C')

    return ee.Image().addBands([t_mean, t_min, t_max, td_c, wspd, dtr,
                                 hours_above_30, hours_above_35, heat_index_approx])

def family_temp_dewpoint_wind_week_and_lags(fc, week_iso):
    if not USE_ERA5_HOURLY:
        return fc
    start = ee.Date(week_iso)
    week_str = ee.Date(week_iso).format('YYYY-MM-dd')

    days = ee.List.sequence(0,6)
    weekly = ee.ImageCollection(days.map(lambda i: _daily_td_w(ee.Date(start).advance(i,'day')))).mean() \
        .rename(['T_mean_week_C','T_min_week_C','T_max_week_C','Td_week_C','wind_speed_week_ms'])

    lags = []
    for k in [1, 2, 3, 5, 7, 10, 14]:
        d = start.advance(-k,'day')
        img = _daily_td_w(d.format('YYYY-MM-dd')).select(
            ['T_mean_C','T_min_C','T_max_C','Td_C','wind_speed_ms'],
            [f'T_mean_d-{k}_C', f'T_min_d-{k}_C', f'T_max_d-{k}_C', f'Td_d-{k}_C', f'wind_speed_d-{k}_ms']
        )
        lags.append(img)

    full = weekly.addBands(ee.ImageCollection(lags).toBands())

    def reduce_one(f):
        r = full.reduceRegion(ee.Reducer.mean(), f.geometry(), SCALE["ERA5"], maxPixels=1e9)
        return f.set(r).set({'week_start': week_str})

    return fc.map(reduce_one)


In [ ]:
def family_water_balance_week(fc, week_iso):
    """NEW: Combined water availability metric"""

    start = ee.Date(week_iso)
    week_str = ee.Date(week_iso).format('YYYY-MM-dd')

    # Precipitation
    precip_week = (ee.ImageCollection(COLL["CHIRPS_DAILY"])
                   .filterDate(start, start.advance(7,'day'))
                   .select("precipitation")
                   .sum())

    # Evaporation (computed as in existing code)
    def _daily_evap(date_iso):
        d0 = ee.Date(date_iso); d1 = d0.advance(1,'day')
        lhf = (ee.ImageCollection(COLL["ERA5_LAND_HOURLY"])
               .filterDate(d0,d1)
               .select("surface_latent_heat_flux").mean())
        SEC_PER_DAY = ee.Number(86400.0)
        LAMBDA = ee.Number(2.45e6)
        W_TO_MM_PER_DAY = SEC_PER_DAY.divide(LAMBDA)
        return lhf.multiply(W_TO_MM_PER_DAY)

    days = ee.List.sequence(0,6)
    evap_week = ee.ImageCollection(days.map(lambda i: _daily_evap(ee.Date(start).advance(i,'day')))).sum()

    # Water balance
    water_deficit = evap_week.subtract(precip_week).rename('water_deficit_mm_week')

    def reduce_one(f):
        r = water_deficit.reduceRegion(ee.Reducer.mean(), f.geometry(), SCALE["ERA5"], maxPixels=1e9)
        return f.set(r).set({'week_start': week_str})

    return fc.map(reduce_one)

In [ ]:
def _daily_evap(date_iso):
    d0 = ee.Date(date_iso); d1 = d0.advance(1,'day')
    lhf = (ee.ImageCollection(COLL["ERA5_LAND_HOURLY"])
           .filterDate(d0,d1)
           .select("surface_latent_heat_flux").mean())
    SEC_PER_DAY = ee.Number(86400.0)
    LAMBDA = ee.Number(2.45e6)
    W_TO_MM_PER_DAY = SEC_PER_DAY.divide(LAMBDA)
    return lhf.multiply(W_TO_MM_PER_DAY).rename('E_mm_day')

def family_evap_week_and_lags(fc, week_iso):
    if not USE_ERA5_EVP:
        return fc
    start = ee.Date(week_iso)
    week_str = ee.Date(week_iso).format('YYYY-MM-dd')

    days = ee.List.sequence(0,6)
    weekly = ee.ImageCollection(days.map(lambda i: _daily_evap(ee.Date(start).advance(i,'day')))).mean() \
        .rename('E_week_mm_per_day')

    lags = []
    for k in [1, 2, 3, 5, 7, 10, 14]:
        d = start.advance(-k,'day')
        lags.append(_daily_evap(d.format('YYYY-MM-dd')).rename(f'E_d-{k}_mm_per_day'))

    full = weekly.addBands(ee.ImageCollection(lags).toBands())

    def reduce_one(f):
        r = full.reduceRegion(ee.Reducer.mean(), f.geometry(), SCALE["ERA5"], maxPixels=1e9)
        return f.set(r).set({'week_start': week_str})

    return fc.map(reduce_one)


In [ ]:
def _daily_sm(date_iso):
    d0 = ee.Date(date_iso); d1 = d0.advance(1,'day')
    day = ee.ImageCollection(COLL["ERA5_LAND_HOURLY"]).filterDate(d0,d1)
    sm1 = day.select("volumetric_soil_water_layer_1").mean().rename('SM1')
    sm2 = day.select("volumetric_soil_water_layer_2").mean().rename('SM2')
    sm3 = day.select("volumetric_soil_water_layer_3").mean().rename('SM3')
    sm4 = day.select("volumetric_soil_water_layer_4").mean().rename('SM4')
    return ee.Image().addBands([sm1, sm2, sm3, sm4]).select(['SM1','SM2','SM3','SM4'])

def family_soilmoist_week_and_lags(fc, week_iso):
    if not USE_ERA5_HOURLY:
        return fc
    start = ee.Date(week_iso)
    week_str = ee.Date(week_iso).format('YYYY-MM-dd')

    days = ee.List.sequence(0,6)
    weekly = ee.ImageCollection(days.map(lambda i: _daily_sm(ee.Date(start).advance(i,'day')))).mean() \
        .rename(['SM1_week','SM2_week','SM3_week','SM4_week'])

    lags = []
    for k in [1, 2, 3, 5, 7, 10, 14]:
        d = start.advance(-k,'day')
        img = _daily_sm(d.format('YYYY-MM-dd')).rename([
            f'SM1_d-{k}', f'SM2_d-{k}', f'SM3_d-{k}', f'SM4_d-{k}'
        ])
        lags.append(img)

    full = weekly.addBands(ee.ImageCollection(lags).toBands())

    def reduce_one(f):
        r = full.reduceRegion(ee.Reducer.mean(), f.geometry(), SCALE["ERA5"], maxPixels=1e9)
        return f.set(r).set({'week_start': week_str})

    return fc.map(reduce_one)


In [ ]:
def family_elevation(fc, week_iso_str=None):
    if not INCLUDE_ELEVATION:
        return fc
    elev = ee.Image(COLL["SRTM"]).select('elevation').rename('elevation_m')
    def reduce_one(f):
        r = elev.reduceRegion(ee.Reducer.mean(), f.geometry(), SCALE["ELEV"], maxPixels=1e9)
        out = {'elevation_m': r.get('elevation_m')}
        if week_iso_str is not None:
            out['week_start'] = week_iso_str
        return f.set(out)
    return fc.map(reduce_one)


## STEP 4 — Map preview

In [ ]:
import folium

def add_ee_layer(folium_map, ee_object, vis_params, name):
    map_id_dict = ee.Image(ee_object).getMapId(vis_params)
    folium.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name=name,
        overlay=True,
        control=True
    ).add_to(folium_map)


In [ ]:
# Initialize map
m = folium.Map(location=[0, 0], zoom_start=2, height=520)

# HSAs
wire = ee.FeatureCollection([
    ee.Feature(p['geom'], {'id': p['id']}) for p in HSA_LIST
])


# CHIRPS weekly sum
wk0 = ee.Date(anchor_mondays[0])
chirps_wk0 = (
    ee.ImageCollection(COLL["CHIRPS_DAILY"])
      .filterDate(wk0, wk0.advance(7, 'day'))
      .select("precipitation")
      .sum()
)

add_ee_layer(
    m,
    chirps_wk0,
    {'min': 0, 'max': 50},
    f'CHIRPS sum {wk0.format("YYYY-MM-dd").getInfo()}'
)

add_ee_layer(
    m,
    wire.style(color='FF0000', fillColor='00000000', width=2),
    {},
    'HSAs'
)


folium.LayerControl().add_to(m)
m


## STEP 5 — Export helpers and per‑polygon driver

In [ ]:
import time
import re

# ============================================
# STEP 5 — Robust driver: compute with cleaned geometry, export WITHOUT geometry
# ============================================

# --- tiny, safe clean-up params (meters)
SIMPLIFY_M = 10
CLEAN_BUFFER_M = 1
MAX_ERROR_M = 10
FALLBACK_BOX_M = 250

# --- Week chunk size for memory management ---
# Reduce this if you still get memory errors; increase for faster exports
WEEKS_PER_CHUNK = 13  # ~3 months of weeks per export task

# --- retry helper for flaky EE compute calls
def _with_retry(fn, label, retries=5, backoff_s=5):
    last_err = None
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            last_err = e
            delay = backoff_s * (2 ** attempt)
            print(f"⚠️  {label} failed (attempt {attempt + 1}/{retries}); retrying in {delay}s: {e}")
            time.sleep(delay)
    raise last_err

def _reduce_region_mean(img, geom, scale, max_pixels=1e9):
    return img.reduceRegion(
        ee.Reducer.mean(),
        geom,
        scale,
        maxPixels=max_pixels,
        bestEffort=True,
        tileScale=4,
    )

def _force_planar(g: ee.Geometry) -> ee.Geometry:
    t = ee.String(g.type())
    coords = g.coordinates()
    poly = ee.Algorithms.If(
        t.equals('Polygon'),
        ee.Geometry.Polygon(coords, None, False),
        ee.Algorithms.If(
            t.equals('MultiPolygon'),
            ee.Geometry.MultiPolygon(coords, None, False),
            g
        )
    )
    return ee.Geometry(poly)

def robust_ee_geom(hid: str) -> ee.Geometry:
    base = ee_geom_by_id(hid)
    g0 = _force_planar(base)
    g0 = g0.buffer(0, MAX_ERROR_M)
    g1 = g0.simplify(SIMPLIFY_M)
    if CLEAN_BUFFER_M and CLEAN_BUFFER_M != 0:
        g1 = g1.buffer(CLEAN_BUFFER_M, MAX_ERROR_M).buffer(-CLEAN_BUFFER_M, MAX_ERROR_M)
    area_ok = ee.Number(g1.area(MAX_ERROR_M)).gt(0)

    def _fallback():
        c = g0.centroid(MAX_ERROR_M)
        return c.buffer(FALLBACK_BOX_M, MAX_ERROR_M).bounds()

    g2 = ee.Geometry(ee.Algorithms.If(area_ok, g1, _fallback()))
    try:
        _ = _with_retry(lambda: g2.type().getInfo(), "geom type")
    except Exception as e:
        print(f"⚠️  Geometry validation failed for {hid!r}, using fallback bounds: {e}")
        g2 = g0.centroid(MAX_ERROR_M).buffer(FALLBACK_BOX_M, MAX_ERROR_M).bounds()
        _ = _with_retry(lambda: g2.type().getInfo(), "geom type fallback")
    return g2

def _feat(props: dict):
    return ee.Feature(None, props)

def _weeks_fc_map(weeks_list, fn_make_props):
    w = ee.List(weeks_list)
    def _mk(d):
        week_str = ee.Date(d).format('YYYY-MM-dd')
        props = fn_make_props(week_str)
        props_dict = ee.Dictionary(props) if props else ee.Dictionary({})
        return ee.Feature(None, props_dict)
    return ee.FeatureCollection(w.map(_mk))

def _precip_props(geom: ee.Geometry, week_iso: ee.String):
    if not USE_CHIRPS:
        return {}
    start = ee.Date(week_iso)
    ic = (ee.ImageCollection(COLL["CHIRPS_DAILY"])
          .filterDate(start, start.advance(7, 'day'))
          .select("precipitation")
          .map(lambda im: im.unmask(0)))

    weekly = ee.Image().addBands([
        ic.mean().rename('P_mean_week'),
        ic.sum().rename('P_total_week'),
        ic.map(lambda im: im.gt(1.0)).mean().rename('wetday_frac_week'),
        ic.map(lambda im: im.gt(20.0)).sum().rename('heavy_days_week'),
        ic.reduce(ee.Reducer.percentile([95])).rename(['P95_week']),
        ic.reduce(ee.Reducer.max()).rename('P_max_day_week'),
    ])

    img = weekly

    for w in [1, 2, 3]:
        d0 = start.advance(-w * 7, 'day')
        d1 = d0.advance(7, 'day')
        ic_lag = (ee.ImageCollection(COLL["CHIRPS_DAILY"])
                  .filterDate(d0, d1)
                  .select("precipitation")
                  .map(lambda im: im.unmask(0)))
        img = img.addBands(ic_lag.sum().rename(f'P_sum_lag_w-{w}'))
        img = img.addBands(ic_lag.mean().rename(f'P_mean_lag_w-{w}'))
        img = img.addBands(ic_lag.map(lambda im: im.gt(10)).sum().rename(f'P_heavy_days_lag_w-{w}'))

    for k in [1, 2, 3, 5, 7, 10, 14]:
        d0 = start.advance(-k, 'day')
        di = (ee.ImageCollection(COLL["CHIRPS_DAILY"])
              .filterDate(d0, d0.advance(1, 'day'))
              .select("precipitation")
              .map(lambda im: im.unmask(0))
              .mean()
              .rename(f'P_d-{k}'))
        img = img.addBands(di)

    r = _reduce_region_mean(img, geom, SCALE["CHIRPS"])

    props = {
        'week_start': week_iso,
        'P_mean_week': r.get('P_mean_week'),
        'P_total_week': r.get('P_total_week'),
        'wetday_frac_week': r.get('wetday_frac_week'),
        'heavy_days_week': r.get('heavy_days_week'),
        'P95_week': r.get('P95_week'),
        'P_max_day_week': r.get('P_max_day_week'),
    }

    for w in [1, 2, 3]:
        props[f'P_sum_lag_w-{w}'] = r.get(f'P_sum_lag_w-{w}')
        props[f'P_mean_lag_w-{w}'] = r.get(f'P_mean_lag_w-{w}')
        props[f'P_heavy_days_lag_w-{w}'] = r.get(f'P_heavy_days_lag_w-{w}')

    for k in [1, 2, 3, 5, 7, 10, 14]:
        props[f'P_d-{k}'] = r.get(f'P_d-{k}')

    return props


def _tdw_props(geom: ee.Geometry, week_iso: ee.String):
    if not USE_ERA5_HOURLY:
        return {}

    def _daily_td_w_improved(date_iso):
        d0 = ee.Date(date_iso)
        d1 = d0.advance(1, 'day')
        day = ee.ImageCollection(COLL["ERA5_LAND_HOURLY"]).filterDate(d0, d1)

        t = day.select("temperature_2m")
        td = day.select("dewpoint_temperature_2m")
        u = day.select("u_component_of_wind_10m")
        v = day.select("v_component_of_wind_10m")

        t_mean = t.mean().subtract(273.15).rename('T_mean_C')
        t_min = t.min().subtract(273.15).rename('T_min_C')
        t_max = t.max().subtract(273.15).rename('T_max_C')
        td_c = td.mean().subtract(273.15).rename('Td_C')
        wspd = u.mean().hypot(v.mean()).rename('wind_speed_ms')
        dtr = t.max().subtract(t.min()).rename('DTR_C')

        hours_above_30 = t.map(lambda img: img.subtract(273.15).gt(30)).sum().rename('hours_above_30C')
        hours_above_35 = t.map(lambda img: img.subtract(273.15).gt(35)).sum().rename('hours_above_35C')

        t_mean_k = t.mean()
        td_mean_k = td.mean()
        heat_index = t_mean_k.add(td_mean_k.subtract(t_mean_k).multiply(0.4)).subtract(273.15).rename('heat_index_C')

        return t_mean.addBands([t_min, t_max, td_c, wspd, dtr, hours_above_30, hours_above_35, heat_index])

    start = ee.Date(week_iso)
    days = ee.List.sequence(0, 6)

    weekly = (ee.ImageCollection(days.map(lambda i: _daily_td_w_improved(ee.Date(start).advance(i, 'day'))))
              .mean()
              .rename([
                  'T_mean_week_C', 'T_min_week_C', 'T_max_week_C', 'Td_week_C',
                  'wind_speed_week_ms', 'DTR_week_C',
                  'hours_above_30C_week', 'hours_above_35C_week', 'heat_index_week_C'
              ]))

    img = weekly

    for k in [1, 2, 3, 5, 7, 10, 14]:
        d = start.advance(-k, 'day')
        lag_img = (_daily_td_w_improved(d.format('YYYY-MM-dd'))
                   .rename([
                       f'T_mean_d-{k}_C', f'T_min_d-{k}_C', f'T_max_d-{k}_C',
                       f'Td_d-{k}_C', f'wind_speed_d-{k}_ms', f'DTR_d-{k}_C',
                       f'hours_above_30C_d-{k}', f'hours_above_35C_d-{k}', f'heat_index_d-{k}_C'
                   ]))
        img = img.addBands(lag_img)

    r = _reduce_region_mean(img, geom, SCALE["ERA5"])

    props = {
        'week_start': week_iso,
        'T_mean_week_C': r.get('T_mean_week_C'),
        'T_min_week_C': r.get('T_min_week_C'),
        'T_max_week_C': r.get('T_max_week_C'),
        'Td_week_C': r.get('Td_week_C'),
        'wind_speed_week_ms': r.get('wind_speed_week_ms'),
        'DTR_week_C': r.get('DTR_week_C'),
        'hours_above_30C_week': r.get('hours_above_30C_week'),
        'hours_above_35C_week': r.get('hours_above_35C_week'),
        'heat_index_week_C': r.get('heat_index_week_C'),
    }

    for k in [1, 2, 3, 5, 7, 10, 14]:
        props[f'T_mean_d-{k}_C'] = r.get(f'T_mean_d-{k}_C')
        props[f'T_min_d-{k}_C'] = r.get(f'T_min_d-{k}_C')
        props[f'T_max_d-{k}_C'] = r.get(f'T_max_d-{k}_C')
        props[f'Td_d-{k}_C'] = r.get(f'Td_d-{k}_C')
        props[f'wind_speed_d-{k}_ms'] = r.get(f'wind_speed_d-{k}_ms')
        props[f'DTR_d-{k}_C'] = r.get(f'DTR_d-{k}_C')
        props[f'hours_above_30C_d-{k}'] = r.get(f'hours_above_30C_d-{k}')
        props[f'hours_above_35C_d-{k}'] = r.get(f'hours_above_35C_d-{k}')
        props[f'heat_index_d-{k}_C'] = r.get(f'heat_index_d-{k}_C')

    return props


def _evap_props(geom: ee.Geometry, week_iso: ee.String):
    if not USE_ERA5_EVP:
        return {}

    def _daily_evap(date_iso):
        d0 = ee.Date(date_iso)
        d1 = d0.advance(1, 'day')
        lhf = (ee.ImageCollection(COLL["ERA5_LAND_HOURLY"])
               .filterDate(d0, d1)
               .select("surface_latent_heat_flux")
               .mean())
        SEC_PER_DAY = ee.Number(86400.0)
        LAMBDA = ee.Number(2.45e6)
        W_TO_MM_PER_DAY = SEC_PER_DAY.divide(LAMBDA)
        return lhf.multiply(W_TO_MM_PER_DAY).rename('E_mm_day')

    start = ee.Date(week_iso)
    days = ee.List.sequence(0, 6)

    weekly = (ee.ImageCollection(days.map(lambda i: _daily_evap(ee.Date(start).advance(i, 'day'))))
              .mean()
              .rename('E_week_mm_per_day'))

    img = weekly

    for k in [1, 2, 3, 5, 7, 10, 14]:
        d = start.advance(-k, 'day')
        lag_img = _daily_evap(d.format('YYYY-MM-dd')).rename(f'E_d-{k}_mm_per_day')
        img = img.addBands(lag_img)

    r = _reduce_region_mean(img, geom, SCALE["ERA5"])

    props = {'week_start': week_iso, 'E_week_mm_per_day': r.get('E_week_mm_per_day')}
    for k in [1, 2, 3, 5, 7, 10, 14]:
        props[f'E_d-{k}_mm_per_day'] = r.get(f'E_d-{k}_mm_per_day')
    return props


def _soil_props(geom: ee.Geometry, week_iso: ee.String):
    if not USE_ERA5_HOURLY:
        return {}

    def _daily_sm(date_iso):
        d0 = ee.Date(date_iso)
        d1 = d0.advance(1, 'day')
        day = ee.ImageCollection(COLL["ERA5_LAND_HOURLY"]).filterDate(d0, d1)
        sm1 = day.select("volumetric_soil_water_layer_1").mean().rename('SM1')
        sm2 = day.select("volumetric_soil_water_layer_2").mean().rename('SM2')
        sm3 = day.select("volumetric_soil_water_layer_3").mean().rename('SM3')
        sm4 = day.select("volumetric_soil_water_layer_4").mean().rename('SM4')
        return sm1.addBands([sm2, sm3, sm4])

    start = ee.Date(week_iso)
    days = ee.List.sequence(0, 6)

    weekly = (ee.ImageCollection(days.map(lambda i: _daily_sm(ee.Date(start).advance(i, 'day'))))
              .mean()
              .rename(['SM1_week', 'SM2_week', 'SM3_week', 'SM4_week']))

    img = weekly

    for k in [1, 2, 3, 5, 7, 10, 14]:
        d = start.advance(-k, 'day')
        lag_img = (_daily_sm(d.format('YYYY-MM-dd'))
                   .rename([f'SM1_d-{k}', f'SM2_d-{k}', f'SM3_d-{k}', f'SM4_d-{k}']))
        img = img.addBands(lag_img)

    r = _reduce_region_mean(img, geom, SCALE["ERA5"])

    props = {
        'week_start': week_iso,
        'SM1_week': r.get('SM1_week'),
        'SM2_week': r.get('SM2_week'),
        'SM3_week': r.get('SM3_week'),
        'SM4_week': r.get('SM4_week'),
    }

    for k in [1, 2, 3, 5, 7, 10, 14]:
        props[f'SM1_d-{k}'] = r.get(f'SM1_d-{k}')
        props[f'SM2_d-{k}'] = r.get(f'SM2_d-{k}')
        props[f'SM3_d-{k}'] = r.get(f'SM3_d-{k}')
        props[f'SM4_d-{k}'] = r.get(f'SM4_d-{k}')

    return props


def _water_props(geom: ee.Geometry, week_iso: ee.String):
    if not (USE_CHIRPS and USE_ERA5_EVP):
        return {}

    start = ee.Date(week_iso)

    precip_week = (ee.ImageCollection(COLL["CHIRPS_DAILY"])
                   .filterDate(start, start.advance(7, 'day'))
                   .select("precipitation")
                   .map(lambda im: im.unmask(0))
                   .sum())

    def _daily_evap(date_iso):
        d0 = ee.Date(date_iso)
        d1 = d0.advance(1, 'day')
        lhf = (ee.ImageCollection(COLL["ERA5_LAND_HOURLY"])
               .filterDate(d0, d1)
               .select("surface_latent_heat_flux")
               .mean())
        SEC_PER_DAY = ee.Number(86400.0)
        LAMBDA = ee.Number(2.45e6)
        W_TO_MM_PER_DAY = SEC_PER_DAY.divide(LAMBDA)
        return lhf.multiply(W_TO_MM_PER_DAY)

    days = ee.List.sequence(0, 6)
    evap_week = ee.ImageCollection(days.map(lambda i: _daily_evap(ee.Date(start).advance(i, 'day')))).sum()

    water_deficit = evap_week.subtract(precip_week).rename('water_deficit_mm_week')
    r = _reduce_region_mean(water_deficit, geom, SCALE["ERA5"])
    return {'week_start': week_iso, 'water_deficit_mm_week': r.get('water_deficit_mm_week')}


def _elev_props(geom: ee.Geometry, week_iso: ee.String):
    if not INCLUDE_ELEVATION:
        return {}
    elev = ee.Image(COLL["SRTM"]).select('elevation').rename('elevation_m')
    r = _reduce_region_mean(elev, geom, SCALE["ELEV"])
    return {'week_start': week_iso, 'elevation_m': r.get('elevation_m')}


def _safe_name(s, max_len=100):
    s = re.sub(r"[^a-zA-Z0-9\.,:;_\-]", "_", str(s))
    return s[:max_len]

def _export_fc_to_drive(fc, description, prefix, drive_folder="CLIMATE_HSA"):
    safe_desc = _safe_name(description)
    safe_prefix = _safe_name(prefix)

    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=safe_desc,
        folder=drive_folder,
        fileNamePrefix=safe_prefix,
        fileFormat="CSV",
    )
    task.start()

    expected_filename = f"{safe_prefix}.csv"
    print(f"✓ Export started: {safe_desc} → Drive/{drive_folder}/{expected_filename}")
    return task, expected_filename


def _chunk_list(lst, chunk_size):
    """Split a list into chunks of at most chunk_size."""
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]


def export_one_polygon_by_family(hsa_id: str, weeks_list, drive_folder="CLIMATE_HSA"):
    """
    Export all enabled families for one polygon.
    Chunks weeks into smaller batches to avoid Earth Engine memory errors.
    Returns (tasks, expected_files) for this polygon.
    """
    tasks = []
    expected_files = []

    geom = robust_ee_geom(hsa_id)

    # Chunk weeks to avoid memory errors on large polygons
    week_chunks = list(_chunk_list(weeks_list, WEEKS_PER_CHUNK))
    use_chunks = len(week_chunks) > 1

    for chunk_idx, chunk_weeks in enumerate(week_chunks):
        chunk_suffix = f"_chunk{chunk_idx}" if use_chunks else ""

        try:
            if USE_CHIRPS:
                fc_p = _weeks_fc_map(chunk_weeks, lambda ws: {id_col: hsa_id, **_precip_props(geom, ws)})
                t, fn = _export_fc_to_drive(fc_p, f"{NETWORK}_precip_{hsa_id}{chunk_suffix}", f"{NETWORK}_HSA_{hsa_id}_precip_lags{chunk_suffix}", drive_folder=drive_folder)
                tasks.append(t); expected_files.append(fn)
        except Exception as e:
            print(f"⚠️  Precip export failed for '{hsa_id}' chunk {chunk_idx}: {e}")

        try:
            if USE_ERA5_HOURLY:
                fc_td_w = _weeks_fc_map(chunk_weeks, lambda ws: {id_col: hsa_id, **_tdw_props(geom, ws)})
                t, fn = _export_fc_to_drive(fc_td_w, f"{NETWORK}_tempdew_wind_{hsa_id}{chunk_suffix}", f"{NETWORK}_HSA_{hsa_id}_tempdew_wind_lags{chunk_suffix}", drive_folder=drive_folder)
                tasks.append(t); expected_files.append(fn)
        except Exception as e:
            print(f"⚠️  Temp/dew/wind export failed for '{hsa_id}' chunk {chunk_idx}: {e}")

        try:
            if USE_ERA5_EVP:
                fc_evp = _weeks_fc_map(chunk_weeks, lambda ws: {id_col: hsa_id, **_evap_props(geom, ws)})
                t, fn = _export_fc_to_drive(fc_evp, f"{NETWORK}_evap_{hsa_id}{chunk_suffix}", f"{NETWORK}_HSA_{hsa_id}_evapERA5_lags{chunk_suffix}", drive_folder=drive_folder)
                tasks.append(t); expected_files.append(fn)
        except Exception as e:
            print(f"⚠️  Evap export failed for '{hsa_id}' chunk {chunk_idx}: {e}")

        try:
            if USE_ERA5_HOURLY:
                fc_sm = _weeks_fc_map(chunk_weeks, lambda ws: {id_col: hsa_id, **_soil_props(geom, ws)})
                t, fn = _export_fc_to_drive(fc_sm, f"{NETWORK}_soilmoist_{hsa_id}{chunk_suffix}", f"{NETWORK}_HSA_{hsa_id}_soilmoistERA5_lags{chunk_suffix}", drive_folder=drive_folder)
                tasks.append(t); expected_files.append(fn)
        except Exception as e:
            print(f"⚠️  Soil moisture export failed for '{hsa_id}' chunk {chunk_idx}: {e}")

        try:
            if USE_CHIRPS and USE_ERA5_EVP:
                fc_water = _weeks_fc_map(chunk_weeks, lambda ws: {id_col: hsa_id, **_water_props(geom, ws)})
                t, fn = _export_fc_to_drive(fc_water, f"{NETWORK}_water_balance_{hsa_id}{chunk_suffix}", f"{NETWORK}_HSA_{hsa_id}_water_balance{chunk_suffix}", drive_folder=drive_folder)
                tasks.append(t); expected_files.append(fn)
        except Exception as e:
            print(f"⚠️  Water balance export failed for '{hsa_id}' chunk {chunk_idx}: {e}")

        try:
            if INCLUDE_ELEVATION:
                fc_el = _weeks_fc_map(chunk_weeks, lambda ws: {id_col: hsa_id, **_elev_props(geom, ws)})
                t, fn = _export_fc_to_drive(fc_el, f"{NETWORK}_elevation_{hsa_id}{chunk_suffix}", f"{NETWORK}_HSA_{hsa_id}_elevation_by_week{chunk_suffix}", drive_folder=drive_folder)
                tasks.append(t); expected_files.append(fn)
        except Exception as e:
            print(f"⚠️  Elevation export failed for '{hsa_id}' chunk {chunk_idx}: {e}")

    return tasks, expected_files

def run_exports_per_polygon(ids=None, start=None, end=None, batch_index=None, batch_size=None,
                            sleep_s=10, retries=3, backoff_s=10, batch_delay_s=None,
                            weeks_start=None, weeks_end=None, weeks_count=None,
                            drive_folder="CLIMATE_HSA"):
    """
    Run exports for a subset of HSAs with pacing and backoff.
    Returns (tasks, expected_files) across the whole run.
    """
    all_tasks = []
    all_expected_files = []

    id_list = _with_retry(
        lambda: ee_fc_hsa.reduceColumns(ee.Reducer.toList(), [id_col]).get("list").getInfo(),
        "fetch id list",
    )

    if TEST_MODE:
        weeks = anchor_mondays[:TEST_WEEK_COUNT]
        ids_use = id_list[:TEST_HSA_COUNT]
        print(f"TEST RUN: {len(ids_use)} polygons × {len(weeks)} weeks")
    else:
        weeks = anchor_mondays
        ids_use = id_list
        print(f"FULL RUN: {len(ids_use)} polygons × {len(weeks)} weeks")

    if weeks_count is not None:
        weeks_start = weeks_start or 0
        weeks_end = weeks_start + weeks_count

    if weeks_start is not None or weeks_end is not None:
        w_start = weeks_start or 0
        w_end = weeks_end if weeks_end is not None else len(weeks)
        weeks = weeks[w_start:w_end]
        print(f"Sliced weeks to indices [{w_start}:{w_end}]: {len(weeks)} weeks")

    if ids is not None:
        ids_use = [hid for hid in ids_use if hid in set(ids)]
        print(f"Filtered by ids: {len(ids_use)} polygons")
    if not ids_use:
        print("⚠️  No polygons matched the provided ids.")
        return [], []

    if batch_index is not None and batch_size is not None:
        start = batch_index * batch_size
        end = start + batch_size

    if start is not None or end is not None:
        start = start or 0
        end = end if end is not None else len(ids_use)
        ids_use = ids_use[start:end]
        print(f"Sliced to indices [{start}:{end}]: {len(ids_use)} polygons")

    for hid in ids_use:
        time.sleep(sleep_s)
        last_err = None
        for attempt in range(retries):
            try:
                tlist, flist = export_one_polygon_by_family(hid, weeks, drive_folder=drive_folder)
                all_tasks.extend(tlist)
                all_expected_files.extend(flist)
                last_err = None
                break
            except Exception as e:
                last_err = e
                delay = backoff_s * (2 ** attempt)
                print(f"⚠️  Retry {attempt + 1}/{retries} for '{hid}' after {delay}s: {e}")
                time.sleep(delay)
        if last_err is not None:
            print(f"⚠️  Skipping '{hid}': {last_err}")

    if batch_delay_s:
        print(f"Batch complete. Sleeping {batch_delay_s}s...")
        time.sleep(batch_delay_s)

    return all_tasks, all_expected_files

print("Driver ready. Call: tasks, expected_files = run_exports_per_polygon(...)")

In [ ]:
import os
from typing import List
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# ----------------------------
# CONFIG
# ----------------------------
DRIVE_FOLDER_NAME = f"CLIMATE_HSA_{BOUNDARY_VERSION.upper()}"  # must match drive_folder in exports
CLIENT_SECRETS_PATH = "client_secrets.json"
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

LOCAL_CLIMATE_DIR = os.path.join(OUT_DIR, f"DRIVE_CLIMATE_BY_HSA_DOWNLOAD_{BOUNDARY_VERSION.upper()}", "FINAL_HSA_CLIMATE")
os.makedirs(LOCAL_CLIMATE_DIR, exist_ok=True)

# ----------------------------
# GOOGLE DRIVE AUTH
# ----------------------------
def get_drive_service(client_secrets_path: str = CLIENT_SECRETS_PATH):
    flow = InstalledAppFlow.from_client_secrets_file(client_secrets_path, SCOPES)
    creds = flow.run_local_server(port=0)
    return build("drive", "v3", credentials=creds)

def get_folder_id(service, folder_name: str) -> str:
    q = (
        "mimeType='application/vnd.google-apps.folder' "
        f"and name='{folder_name}' and trashed=false"
    )
    res = service.files().list(q=q, fields="files(id,name)").execute()
    files = res.get("files", [])
    if not files:
        raise FileNotFoundError(f"Drive folder '{folder_name}' not found.")
    if len(files) > 1:
        print(f"Warning: multiple folders named '{folder_name}'. Using the first match.")
    return files[0]["id"]

def list_files_in_folder(service, folder_id: str):
    q = f"'{folder_id}' in parents and trashed=false"
    page_token = None
    out = []
    while True:
        res = service.files().list(
            q=q,
            pageToken=page_token,
            fields="nextPageToken, files(id,name,modifiedTime,size)",
            pageSize=1000,
        ).execute()
        out.extend(res.get("files", []))
        page_token = res.get("nextPageToken")
        if not page_token:
            break
    return out

drive_service = get_drive_service(CLIENT_SECRETS_PATH)
try:
    drive_folder_id = get_folder_id(drive_service, DRIVE_FOLDER_NAME)
    print("✓ Drive API authenticated.")
    print("✓ Drive folder:", DRIVE_FOLDER_NAME, "id:", drive_folder_id)
except FileNotFoundError:
    drive_folder_id = None
    print("✓ Drive API authenticated.")
    print(f"  Note: Drive folder '{DRIVE_FOLDER_NAME}' not found yet — GEE will create it when exports run.")
    print(f"  Run the export cell to trigger exports; folder ID will be resolved automatically.")


In [ ]:
import os

def download_drive_file(service, file_id: str, local_path: str):
    request = service.files().get_media(fileId=file_id)
    with open(local_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return local_path

def download_expected_files(
    drive_service,
    folder_id: str,
    expected_files: List[str],
    local_dir: str = LOCAL_CLIMATE_DIR,
    allow_fuzzy: bool = False,
):
    """Download expected files from Drive.

    allow_fuzzy=True: if an exact match is missing, accept any Drive file whose
    name shares the same stem and extension (handles GEE deduplication suffixes
    like ' (1)').  The file is always saved under the canonical expected name.
    """
    expected = set(expected_files)
    os.makedirs(local_dir, exist_ok=True)

    drive_files = list_files_in_folder(drive_service, folder_id)
    name_to_id = {f["name"]: f["id"] for f in drive_files}

    resolved = {}
    for name in sorted(expected):
        if name in name_to_id:
            resolved[name] = name
        elif allow_fuzzy:
            stem = name.rsplit(".", 1)[0] if "." in name else name
            ext  = "." + name.rsplit(".", 1)[1] if "." in name else ""
            candidates = sorted(
                n for n in name_to_id if n.startswith(stem) and n.endswith(ext)
            )
            if candidates:
                resolved[name] = candidates[-1]
                print(f"  Fuzzy match: {name!r} -> {candidates[-1]!r}")

    still_missing = sorted(expected - set(resolved.keys()))
    if still_missing:
        raise FileNotFoundError(
            f"These expected files are still missing on Drive: {still_missing[:10]}"
            + (" ..." if len(still_missing) > 10 else "")
        )

    for name in sorted(expected):
        actual = resolved[name]
        local_path = os.path.join(local_dir, name)
        download_drive_file(drive_service, name_to_id[actual], local_path)
        if actual != name:
            print(f"\u2713 Downloaded {actual!r} -> {local_path}  (fuzzy for {name!r})")
        else:
            print(f"\u2713 Downloaded {name} -> {local_path}")

    print(f"\u2713 All files downloaded to: {os.path.abspath(local_dir)}")


## STEP 6 — Processing

In [ ]:
from datetime import datetime
import time

def wait_for_exports_and_drive_files(
    tasks: List,
    expected_files: List[str],
    drive_service,
    drive_folder_id: str,
    poll_seconds: int = 60,
    max_stall_polls: int = 10,
):
    """Poll GEE tasks and Drive until all expected files appear.

    If all tasks reach COMPLETED but expected files are still missing after
    max_stall_polls consecutive polls, searches Drive for fuzzy name matches
    (e.g. ' (1)' deduplication suffixes) and raises a descriptive error so
    the caller can decide whether to re-export or use download_expected_files
    with allow_fuzzy=True.
    """
    expected = set(expected_files)
    consecutive_errors = 0
    stall_polls = 0

    while True:
        try:
            statuses = [t.status() for t in tasks]
            states = [s.get("state", "UNKNOWN") for s in statuses]
            n_completed = sum(st == "COMPLETED" for st in states)
            n_total = len(tasks)
            failed = [s for s in statuses if s.get("state") in {"FAILED", "CANCELLED"}]

            drive_files = list_files_in_folder(drive_service, drive_folder_id)
            drive_names = set(f["name"] for f in drive_files)

            have = len(drive_names & expected)
            missing = sorted(expected - drive_names)
            consecutive_errors = 0

        except Exception as e:
            consecutive_errors += 1
            now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"[{now}] \u26a0  Poll error (attempt {consecutive_errors}): {type(e).__name__}: {e}")
            if consecutive_errors >= 5:
                raise RuntimeError(f"Too many consecutive poll errors \u2014 last: {e}") from e
            time.sleep(poll_seconds)
            continue

        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{now}] Tasks completed: {n_completed}/{n_total} | Drive files present: {have}/{len(expected)}")
        if missing:
            print(f"  Missing (up to 5): {missing[:5]}" + (" ..." if len(missing) > 5 else ""))

        if failed:
            msg_lines = ["One or more Earth Engine exports failed/cancelled:"]
            for s in failed:
                msg_lines.append(
                    f" - {s.get('state')} : {s.get('description','')} | {s.get('error_message','')}"
                )
            raise RuntimeError("\n".join(msg_lines))

        if not missing:
            print(f"[{now}] \u2713 All {len(expected)} expected files are visible on Drive.")
            if n_completed < n_total:
                print(f"  (Note: {n_total - n_completed} GEE task(s) still show non-COMPLETED; files are present.)")
            return

        # All tasks done but file(s) still missing \u2014 start counting stall polls
        if n_completed == n_total:
            stall_polls += 1
            print(f"[{now}]   All tasks COMPLETED; waiting for Drive to reflect {len(missing)} file(s) "
                  f"(stall poll {stall_polls}/{max_stall_polls}).")

            if stall_polls >= max_stall_polls:
                fuzzy = {}
                for mf in sorted(missing):
                    stem = mf.rsplit(".", 1)[0] if "." in mf else mf
                    ext  = "." + mf.rsplit(".", 1)[1] if "." in mf else ""
                    candidates = sorted(
                        f["name"] for f in drive_files
                        if f["name"].startswith(stem) and f["name"].endswith(ext)
                    )
                    if candidates:
                        fuzzy[mf] = candidates

                print(f"\n[{now}] \u26a0  Stalled: all {n_total} GEE tasks COMPLETED but "
                      f"{len(missing)} expected file(s) absent after {stall_polls} extra polls.")
                for mf in sorted(missing):
                    cands = fuzzy.get(mf, [])
                    if cands:
                        print(f"  {mf!r}  ->  possible Drive match(es): {cands}")
                    else:
                        print(f"  {mf!r}  ->  no fuzzy match found in Drive folder")

                hint = (
                    "Likely cause: GEE wrote a duplicate file with a ' (N)' suffix "
                    "(Drive deduplication when the file already existed).\n"
                    "Options:\n"
                    "  1. Call download_expected_files(..., allow_fuzzy=True) to download the fuzzy match.\n"
                    "  2. Re-export only the missing HSA(s) via run_exports_per_polygon(ids=[...]).\n"
                    "  3. Check Drive manually for the file under a slightly different name."
                )
                raise RuntimeError(
                    f"Stalled: all {n_total} GEE tasks COMPLETED but {len(missing)} expected file(s) "
                    f"still absent from Drive after {stall_polls} extra polls.\n"
                    f"Missing: {sorted(missing)[:10]}\n{hint}"
                )
        else:
            stall_polls = 0

        time.sleep(poll_seconds)


In [ ]:
TEST_MODE = False

# Launch exports only if not already in memory (safe to re-run after a kernel interrupt)
if "tasks" not in dir() or not tasks:
    tasks, expected_files = run_exports_per_polygon(drive_folder=DRIVE_FOLDER_NAME)
    print(f"\u2713 Launched {len(tasks)} export tasks, expecting {len(expected_files)} files in Drive/{DRIVE_FOLDER_NAME}")
else:
    print(f"Reusing existing {len(tasks)} tasks / {len(expected_files)} expected files from memory.")

if drive_folder_id is None:
    import time as _time
    print(f"  Waiting 60s for GEE to create Drive folder '{DRIVE_FOLDER_NAME}'...")
    _time.sleep(60)
    drive_folder_id = get_folder_id(drive_service, DRIVE_FOLDER_NAME)
    print(f"  \u2713 Drive folder resolved: {drive_folder_id}")

# Polls until all files appear; raises with fuzzy-match hints if all tasks complete
# but a file is still missing after max_stall_polls * poll_seconds seconds.
# Remove stale chunk files from prior interrupted runs so find_chunk_groups
# only sees chunks from the current download.
import pathlib as _pl, re as _re
_chunk_pat = _re.compile(r'_chunk\d+\.csv$')
_stale = [p for p in _pl.Path(LOCAL_CLIMATE_DIR).glob("*.csv") if _chunk_pat.search(p.name)]
if _stale:
    print(f"Removing {len(_stale)} stale chunk file(s) from prior run...")
    for _p in _stale:
        _p.unlink()

wait_for_exports_and_drive_files(
    tasks=tasks,
    expected_files=expected_files,
    drive_service=drive_service,
    drive_folder_id=drive_folder_id,
    poll_seconds=60,
    max_stall_polls=10,
)

# allow_fuzzy=True handles GEE Drive deduplication (e.g. ' (1)' suffix)
download_expected_files(
    drive_service=drive_service,
    folder_id=drive_folder_id,
    expected_files=expected_files,
    local_dir=LOCAL_CLIMATE_DIR,
    allow_fuzzy=True,
)


## STEP 7 — Concatenate Chunked Files

When exports are chunked (due to `WEEKS_PER_CHUNK`), multiple CSV files are created per HSA per data family. This step:
1. Identifies chunked file groups by pattern matching
2. Concatenates chunks in order, validating week continuity
3. Checks for duplicates and missing weeks
4. Saves merged files and removes chunk files

In [ ]:
import pandas as pd
import re
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

# ============================================
# STEP 7 — Concatenate chunked CSV files with validation
# ============================================

def find_chunk_groups(directory: str) -> Dict[str, List[Tuple[int, str]]]:
    """
    Find all chunked file groups in a directory.
    
    Returns dict mapping base_name -> [(chunk_idx, filepath), ...]
    e.g. "NCD_HSA_Facility_X_precip_lags" -> [(0, "...chunk0.csv"), (1, "...chunk1.csv")]
    
    Also includes non-chunked files as single-element lists with chunk_idx=-1.
    """
    directory = Path(directory)
    chunk_pattern = re.compile(r'^(.+)_chunk(\d+)\.csv$')
    
    groups = defaultdict(list)
    non_chunked = []
    
    for f in directory.glob("*.csv"):
        match = chunk_pattern.match(f.name)
        if match:
            base_name = match.group(1)
            chunk_idx = int(match.group(2))
            groups[base_name].append((chunk_idx, str(f)))
        else:
            # Non-chunked file
            base_name = f.stem  # filename without .csv
            non_chunked.append((base_name, str(f)))
    
    # Sort each group by chunk index
    for base_name in groups:
        groups[base_name].sort(key=lambda x: x[0])
    
    return dict(groups), non_chunked


def validate_week_continuity(df: pd.DataFrame, base_name: str) -> List[str]:
    """
    Validate that weeks are continuous and properly ordered.
    Returns list of warning messages (empty if all OK).
    """
    warnings = []
    
    if 'week_start' not in df.columns:
        warnings.append(f"{base_name}: Missing 'week_start' column")
        return warnings
    
    # Convert to datetime for validation
    df_sorted = df.sort_values('week_start').copy()
    weeks = pd.to_datetime(df_sorted['week_start'])
    
    # Check for duplicates
    duplicates = df_sorted[df_sorted.duplicated(subset=['week_start'], keep=False)]
    if len(duplicates) > 0:
        dup_weeks = duplicates['week_start'].unique().tolist()
        warnings.append(f"{base_name}: Found {len(dup_weeks)} duplicate week(s): {dup_weeks[:5]}")
    
    # Check for gaps (expecting 7-day intervals)
    diffs = weeks.diff().dropna()
    expected_diff = pd.Timedelta(days=7)
    gaps = diffs[diffs != expected_diff]
    if len(gaps) > 0:
        gap_indices = gaps.index.tolist()
        gap_details = []
        for idx in gap_indices[:5]:  # Show first 5 gaps
            prev_week = df_sorted.loc[idx - 1, 'week_start'] if idx > 0 else 'N/A'
            curr_week = df_sorted.loc[idx, 'week_start']
            gap_details.append(f"{prev_week} -> {curr_week} ({diffs.loc[idx].days}d)")
        warnings.append(f"{base_name}: Found {len(gaps)} week gap(s): {gap_details}")
    
    return warnings


def validate_schema_consistency(dfs: List[pd.DataFrame], base_name: str) -> List[str]:
    """
    Validate that all chunk DataFrames have the same columns.
    Returns list of warning messages.
    """
    warnings = []
    if len(dfs) < 2:
        return warnings
    
    reference_cols = set(dfs[0].columns)
    for i, df in enumerate(dfs[1:], start=1):
        current_cols = set(df.columns)
        if current_cols != reference_cols:
            missing = reference_cols - current_cols
            extra = current_cols - reference_cols
            if missing:
                warnings.append(f"{base_name} chunk {i}: Missing columns: {missing}")
            if extra:
                warnings.append(f"{base_name} chunk {i}: Extra columns: {extra}")
    
    return warnings


def concatenate_chunks(
    directory: str,
    output_dir: Optional[str] = None,
    remove_chunks: bool = True,
    expected_weeks: Optional[List[str]] = None,
) -> Dict[str, any]:
    """
    Concatenate all chunked CSV files in a directory.
    
    Args:
        directory: Path to directory containing chunked CSVs
        output_dir: Where to save merged files (default: same as input)
        remove_chunks: Whether to delete chunk files after successful merge
        expected_weeks: Optional list of expected week_start values for validation
    
    Returns:
        Dict with statistics and any validation warnings
    """
    directory = Path(directory)
    output_dir = Path(output_dir) if output_dir else directory
    output_dir.mkdir(parents=True, exist_ok=True)
    
    chunk_groups, non_chunked = find_chunk_groups(directory)
    
    results = {
        'merged_files': [],
        'skipped_non_chunked': [],
        'warnings': [],
        'errors': [],
        'stats': {
            'total_chunk_groups': len(chunk_groups),
            'total_non_chunked': len(non_chunked),
            'total_chunks_processed': 0,
            'total_rows_merged': 0,
        }
    }
    
    # Process each chunk group
    for base_name, chunks in chunk_groups.items():
        chunk_indices = [c[0] for c in chunks]
        chunk_files = [c[1] for c in chunks]
        
        # Validate chunk indices are sequential starting from 0
        expected_indices = list(range(len(chunks)))
        if chunk_indices != expected_indices:
            results['errors'].append(
                f"{base_name}: Non-sequential chunk indices: {chunk_indices} (expected {expected_indices})"
            )
            continue
        
        try:
            # Load all chunks
            dfs = []
            for chunk_file in chunk_files:
                df = pd.read_csv(chunk_file)
                dfs.append(df)
                results['stats']['total_chunks_processed'] += 1
            
            # Validate schema consistency
            schema_warnings = validate_schema_consistency(dfs, base_name)
            results['warnings'].extend(schema_warnings)
            
            # Concatenate
            merged = pd.concat(dfs, ignore_index=True)
            
            # Sort by week_start if present
            if 'week_start' in merged.columns:
                merged = merged.sort_values('week_start').reset_index(drop=True)
            
            # Validate week continuity
            continuity_warnings = validate_week_continuity(merged, base_name)
            results['warnings'].extend(continuity_warnings)
            
            # Check against expected weeks if provided
            if expected_weeks and 'week_start' in merged.columns:
                actual_weeks = set(merged['week_start'].astype(str))
                expected_set = set(expected_weeks)
                missing_weeks = expected_set - actual_weeks
                extra_weeks = actual_weeks - expected_set
                if missing_weeks:
                    results['warnings'].append(
                        f"{base_name}: Missing {len(missing_weeks)} expected week(s)"
                    )
                if extra_weeks:
                    results['warnings'].append(
                        f"{base_name}: Found {len(extra_weeks)} unexpected week(s)"
                    )
            
            # Remove duplicates if any (keep first occurrence)
            if 'week_start' in merged.columns:
                before_dedup = len(merged)
                merged = merged.drop_duplicates(subset=['week_start'], keep='first')
                after_dedup = len(merged)
                if before_dedup != after_dedup:
                    results['warnings'].append(
                        f"{base_name}: Removed {before_dedup - after_dedup} duplicate row(s)"
                    )
            
            # Save merged file
            output_file = output_dir / f"{base_name}.csv"
            merged.to_csv(output_file, index=False)
            results['merged_files'].append(str(output_file))
            results['stats']['total_rows_merged'] += len(merged)
            
            print(f"✓ Merged {len(chunks)} chunks → {output_file.name} ({len(merged)} rows)")
            
            # Remove chunk files if requested
            if remove_chunks:
                for chunk_file in chunk_files:
                    Path(chunk_file).unlink()
                print(f"  Removed {len(chunk_files)} chunk file(s)")
                
        except Exception as e:
            results['errors'].append(f"{base_name}: {str(e)}")
            print(f"✗ Error processing {base_name}: {e}")
    
    # Report non-chunked files
    for base_name, filepath in non_chunked:
        results['skipped_non_chunked'].append(filepath)
    
    if non_chunked:
        print(f"\n✓ {len(non_chunked)} non-chunked file(s) unchanged")
    
    return results


def validate_final_dataset(
    directory: str,
    expected_weeks: List[str],
    id_col: str = "FacilityName",
) -> Dict[str, any]:
    """
    Validate the final merged dataset for completeness.
    
    Args:
        directory: Path to directory containing merged CSVs
        expected_weeks: List of expected week_start values
        id_col: Column name for HSA identifier
    
    Returns:
        Validation report dict
    """
    directory = Path(directory)
    
    report = {
        'files_checked': 0,
        'total_rows': 0,
        'issues': [],
        'summary_by_family': {},
    }
    
    # Group files by family type
    family_patterns = {
        'precip': r'_precip_lags\.csv$',
        'tempdew_wind': r'_tempdew_wind_lags\.csv$',
        'evap': r'_evapERA5_lags\.csv$',
        'soilmoist': r'_soilmoistERA5_lags\.csv$',
        'water_balance': r'_water_balance\.csv$',
        'elevation': r'_elevation_by_week\.csv$',
    }
    
    for family, pattern in family_patterns.items():
        family_files = [f for f in directory.glob("*.csv") if re.search(pattern, f.name)]
        
        if not family_files:
            report['issues'].append(f"No files found for family: {family}")
            continue
        
        family_stats = {
            'file_count': len(family_files),
            'total_rows': 0,
            'hsa_count': 0,
            'weeks_per_hsa': {},
        }
        
        for f in family_files:
            try:
                df = pd.read_csv(f)
                report['files_checked'] += 1
                report['total_rows'] += len(df)
                family_stats['total_rows'] += len(df)
                
                if id_col in df.columns and 'week_start' in df.columns:
                    hsas = df[id_col].unique()
                    family_stats['hsa_count'] = len(hsas)
                    
                    for hsa in hsas:
                        hsa_weeks = df[df[id_col] == hsa]['week_start'].unique()
                        family_stats['weeks_per_hsa'][hsa] = len(hsa_weeks)
                        
                        # Check for missing weeks
                        missing = set(expected_weeks) - set(hsa_weeks.astype(str))
                        if missing:
                            report['issues'].append(
                                f"{family}/{hsa}: Missing {len(missing)} week(s)"
                            )
                            
            except Exception as e:
                report['issues'].append(f"Error reading {f.name}: {e}")
        
        report['summary_by_family'][family] = family_stats
    
    return report


# ============================================
# Run concatenation
# ============================================

print("=" * 60)
print("STEP 7: Concatenating chunked CSV files")
print("=" * 60)

concat_results = concatenate_chunks(
    directory=LOCAL_CLIMATE_DIR,
    output_dir=LOCAL_CLIMATE_DIR,
    remove_chunks=True,  # Set False to keep chunk files for debugging
    expected_weeks=anchor_mondays if not TEST_MODE else anchor_mondays[:TEST_WEEK_COUNT],
)

print("\n" + "=" * 60)
print("Concatenation Summary")
print("=" * 60)
print(f"Chunk groups merged: {concat_results['stats']['total_chunk_groups']}")
print(f"Total chunks processed: {concat_results['stats']['total_chunks_processed']}")
print(f"Total rows in merged files: {concat_results['stats']['total_rows_merged']}")
print(f"Non-chunked files (unchanged): {concat_results['stats']['total_non_chunked']}")

if concat_results['warnings']:
    print(f"\n⚠️  Warnings ({len(concat_results['warnings'])}):")
    for w in concat_results['warnings'][:10]:
        print(f"   - {w}")
    if len(concat_results['warnings']) > 10:
        print(f"   ... and {len(concat_results['warnings']) - 10} more")

if concat_results['errors']:
    print(f"\n❌ Errors ({len(concat_results['errors'])}):")
    for e in concat_results['errors']:
        print(f"   - {e}")

# ============================================
# Final validation
# ============================================

print("\n" + "=" * 60)
print("Final Dataset Validation")
print("=" * 60)

validation = validate_final_dataset(
    directory=LOCAL_CLIMATE_DIR,
    expected_weeks=anchor_mondays if not TEST_MODE else anchor_mondays[:TEST_WEEK_COUNT],
    id_col=id_col,
)

print(f"Files checked: {validation['files_checked']}")
print(f"Total rows: {validation['total_rows']}")

if validation['summary_by_family']:
    print("\nBy data family:")
    for family, stats in validation['summary_by_family'].items():
        print(f"  {family}: {stats['file_count']} file(s), {stats['total_rows']} rows, {stats['hsa_count']} HSA(s)")

if validation['issues']:
    print(f"\n⚠️  Validation issues ({len(validation['issues'])}):")
    for issue in validation['issues'][:10]:
        print(f"   - {issue}")
    if len(validation['issues']) > 10:
        print(f"   ... and {len(validation['issues']) - 10} more")
else:
    print("\n✓ All validations passed!")

print(f"\n✓ Final data available at: {os.path.abspath(LOCAL_CLIMATE_DIR)}")

In [ ]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "GEE_local_HSA_Weekly_Climate_Lagged_chunked"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', ''))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_DIR', globals().get('OUT_ROOT', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', 'out')))))
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
BOUNDARY_VERSION = globals().get('BOUNDARY_VERSION', os.environ.get('BOUNDARY_VERSION', os.environ.get('PIPELINE_VERSION', 'v7')))
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_{BOUNDARY_VERSION}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
]
if HSA_MODE:
    meta.append(f"- HSA mode: {HSA_MODE}")
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")
